# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library and Croissant schema. All dataset elements (record sets, fields, columns) are referenced by their unique `@id`s for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their unique `@id`s using the dataset's metadata.

We'll programmatically list record sets and their fields by their `@id` for reference.

In [ ]:
# List all record sets and print their fields/columns by @id
record_sets = metadata.record_sets

print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"- Record Set name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")

    # List fields by @id within the record set
    fields = rs.fields if hasattr(rs, 'fields') else []
    if fields:
        print("  Fields:")
        for field in fields:
            field_name = getattr(field, 'name', 'N/A')
            field_id = getattr(field, 'id', None)
            print(f"    - {field_name}, @id: {field_id}")
    print()

## 3. Data Extraction
We'll extract data from one or more record sets into pandas DataFrames. All references are to be made by the record set and field `@id`s as listed above.

Let's dynamically extract the data from every available record set.

In [ ]:
# Build a DataFrame for each record set referenced by its @id
dataframes = {}
record_set_ids = [rs.id for rs in metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded record set: {record_set_id}, #rows: {len(df)}")
    print(f"Columns (@id): {list(df.columns)}\n")
    print(df.head(3))
    print("-"*60)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing to one record set. We'll select the first record set (by `@id`), choose a numeric field (by `@id`), and perform basic filtering, normalization, and grouping.

In [ ]:
# For demonstration, pick the first record set
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Selected Record Set: {record_set_id}")
    
    # Attempt to select a numeric field by inspecting dtypes or column names
    # If the dataset has field metadata, we could use it; here, check data
    numeric_col = None
    for col in df.columns:
        # Check if column looks numeric (try conversion)
        try:
            pd.to_numeric(df[col].dropna().iloc[:10])
            numeric_col = col
            break
        except:
            continue
    if numeric_col is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Numeric field selected (by @id): {numeric_col}")
        # Convert field to numeric
        df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')

        # Filter rows with value > threshold
        threshold = df[numeric_col].quantile(0.75) if df[numeric_col].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_col] > threshold]
        print(f"\nFiltered records with {numeric_col} > {threshold} (top quartile):")
        print(filtered_df[[numeric_col]].head())

        # Normalize numeric column
        filtered_df[numeric_col + '_normalized'] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"\nNormalized '{numeric_col}' for filtered records:")
        print(filtered_df[[numeric_col, numeric_col + '_normalized']].head())

        # Attempt to group by a categorical field (choose non-numeric field)
        group_field = None
        for col in df.columns:
            if col != numeric_col and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_col].mean().to_frame()
            print(f"\nMean of {numeric_col} grouped by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field (if available) from the example record set, and relationship with a group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if record_set_ids and numeric_col is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_col].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_col} (@id)")
    plt.xlabel(numeric_col)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field, if available
    if group_field:
        ngroups = df[group_field].nunique()
        if ngroups < 20:
            plt.figure(figsize=(10,4))
            sns.boxplot(data=df, x=group_field, y=numeric_col)
            plt.title(f"{numeric_col} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No numeric field or record set to display visualization.")

## 6. Conclusion
We successfully loaded the FAIR^2 dataset using `mlcroissant`, explored its structure (record sets and associated fields by their `@id`s), and performed basic data extraction and exploratory analysis on the available records. This workflow demonstrates a reproducible approach for FAIR dataset exploration leveraging the Croissant schema, ensuring precise field references and enabling further downstream analysis.